# Lesson 4.2: Standard Error

Compute the standard error of a mean by formula, then check it against simulated sample means and watch it shrink as n grows.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. SE by formula

The lesson's eight midterm scores. The standard error is the sample SD (ddof=1) divided by the square root of n. It says how precisely this class mean is pinned down.

In [1]:
import numpy as np

scores = np.array([72, 85, 68, 91, 77, 83, 64, 80])
n = len(scores)
mean = scores.mean()
s = scores.std(ddof=1)
se = s / np.sqrt(n)

print(f'mean = {mean:.2f}')
print(f's    = {s:.2f}')
print(f'SE   = {se:.2f}')

mean = 77.50
s    = 9.09
SE   = 3.21


## 2. A population to test it on

To check the formula we need many samples, so we build a skewed population (true mean 30, true SD 15) and draw a lot of samples from it.

In [2]:
rng = np.random.default_rng(7)
population = 15 + rng.exponential(15, size=100_000)
sigma = population.std()
print(f'population SD = {sigma:.2f}')

population SD = 14.89


## 3. SE shrinks as n grows

For each n we draw 2,000 samples and take the SD of their means. That should match sigma / sqrt(n), and also the SE we would get from a single sample. Each time n is multiplied by 4, the SE halves.

In [3]:
print('   n   SD of 2000 means   sigma/sqrt(n)   s/sqrt(n), one sample')
for n in (10, 40, 160, 640):
    means = rng.choice(population, size=(2000, n)).mean(axis=1)
    one = rng.choice(population, size=n)
    print(f'{n:4d}   {means.std(ddof=1):14.2f}   {sigma / np.sqrt(n):13.2f}   {one.std(ddof=1) / np.sqrt(n):12.2f}')

   n   SD of 2000 means   sigma/sqrt(n)   s/sqrt(n), one sample
  10             4.79            4.71           4.09
  40             2.36            2.35           2.42
 160             1.20            1.18           1.25
 640             0.57            0.59           0.55


## 4. Plan a sample size

To get an SE of 1 when sigma is about 15, solve 15 / sqrt(n) = 1. Always round the answer up.

In [4]:
target_se = 1
sigma_guess = 15
n_needed = int(np.ceil((sigma_guess / target_se) ** 2))
print('sample size needed:', n_needed)

sample size needed: 225


## Try this

- Add a ninth score of 40 and predict whether the SE goes up or down, then run it.
- In the simulation loop, change the sizes to (25, 100, 400) and check the SE halves each time.
- Change sigma_guess to 30 and predict the sample size needed for an SE of 1.

## The same idea in R, SPSS and Stata

**R**

```r
scores <- c(72, 85, 68, 91, 77, 83, 64, 80)
mean(scores)
sd(scores)
sd(scores) / sqrt(length(scores))
set.seed(7)
for (n in c(10, 40, 160, 640)) print(sd(replicate(2000, mean(15 + rexp(n, 1/15)))))
```

**SPSS** (Analyze > Descriptive Statistics > Descriptives > Options > S.E. mean)

```
DATA LIST FREE / score.
BEGIN DATA
72 85 68 91 77 83 64 80
END DATA.
DESCRIPTIVES VARIABLES=score
  /STATISTICS=MEAN SEMEAN STDDEV.
```

**Stata**

```
clear
input score
72
85
68
91
77
83
64
80
end
ci means score
summarize score
display r(sd) / sqrt(r(N))
```